In [0]:
dbutils.widgets.text("source_system","","Enter source_system")
dbutils.widgets.text("table_name","","Enter Table_name")
dbutils.widgets.text("landing_time","","Enter time")
dbutils.widgets.text("source_file_format","")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_time = dbutils.widgets.get("landing_time")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
base_landing_path = "/Volumes/adb_databricks_careflow/landing/volume/" 
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_time}/"

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
    
missing_path = None
paths = [source_path, table_path, landing_path]
for path in paths:
    if not path_exists(path):
        missing_path=path
        break

if missing_path:
    message = (
        f"No new files are received for source_system={source_system}, "
        f"table_name={table_name}, landing_timestamp={landing_time}."
    )
    print(message)
    dbutils.notebook.exit(message)

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format ("csv").option("header","true").option("inferSchema","true").load(landing_path)

else:
    landing_df = spark.read.format ("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F
updated_df = landing_df.withColumn("_insert_time",F.current_timestamp()).withColumn("landing_time",F.lit(landing_time))
                              

### This is the bronze layer so we used write mode is `append` so that we can have all data and its history.
##### FULL load 

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_databricks_careflow.bronze.{table_name}")